# 实验 19 — dbt 自定义测试

**目标**：内置的 `not_null` / `unique` / `accepted_values` / `relationships` 不够用时，自己写测试。

## 两种自定义测试

| 类型 | 位置 | 何时用 |
|---|---|---|
| **singular test** | `tests/<name>.sql` | 一次性的业务规则，不需要复用 |
| **generic test** | `macros/test_<name>.sql` 或 `tests/generic/<name>.sql` | 跨模型复用，参数化 |

规则：**test 应该 select 那些违反不变量的行**。返回 0 行 = 通过，>0 行 = 失败。

本仓库的样例：
- [tests/assert_positive_rates.sql](../dbt_project/tests/assert_positive_rates.sql) —— singular，检查 rate > 0
- [macros/test_no_future_dates.sql](../dbt_project/macros/test_no_future_dates.sql) —— generic，检查列里没有未来日期

In [ ]:
import subprocess, os
def dbt(*a):
    r = subprocess.run(['uv','run','dbt',*a], capture_output=True, text=True,
                       cwd='../dbt_project',
                       env={**os.environ,'DBT_PROFILES_DIR':'.'})
    return r.stdout + r.stderr

# 先确保模型是新的
print(dbt('build','--exclude','snp_currencies')[-1500:])

In [ ]:
# 把 generic test 应用到 stg_ecb_rates.rate_date
# 临时编辑 _staging.yml 加上 no_future_dates
from pathlib import Path
yml = Path('../dbt_project/models/staging/_staging.yml')
orig = yml.read_text()
yml.write_text(orig.replace(
    '- name: rate_date\n        tests:\n          - not_null',
    '- name: rate_date\n        tests:\n          - not_null\n          - no_future_dates'
))
print(yml.read_text())

In [ ]:
# 跑测试 —— 应该全通过
print(dbt('test','--select','stg_ecb_rates')[-1500:])

In [ ]:
# 故意造一条未来日期数据，验证 generic test 真的能抓到
import duckdb
con = duckdb.connect('../data/sandbox.duckdb')
con.execute("insert into raw_ecb.daily_rates (date,currency,rate,_dlt_load_id,_dlt_id) values ('2099-01-01','USD',1.0,'manual','aaa')")
# stg 是 view，立刻反映
print('future row in stg?',
      con.execute("select * from main.stg_ecb_rates where rate_date='2099-01-01'").fetchall())
print()
print(dbt('test','--select','stg_ecb_rates')[-1500:])

In [ ]:
# 清理：删掉造假数据 + 还原 _staging.yml
con.execute("delete from raw_ecb.daily_rates where date='2099-01-01' and _dlt_load_id='manual'")
Path('../dbt_project/models/staging/_staging.yml').write_text(orig)
print(dbt('test','--select','stg_ecb_rates')[-500:])

In [ ]:
# Singular test 已经在 tests/assert_positive_rates.sql。dbt build 会自动跑
print(dbt('test','--select','assert_positive_rates')[-800:])

## 生产环境踩坑提示

- **测试严重度**：`severity: warn` vs `error`。新接入的数据源经常用 warn 先观察一段时间，稳定后再升级到 error。
  ```yaml
  tests:
    - no_future_dates:
        config:
          severity: warn
          warn_if: '> 0'
          error_if: '> 100'
  ```
- **失败行存到 audit 表**：`dbt test --store-failures` 把违反不变量的行写到 `dbt_test__audit` schema，便于事后调查。生产里强烈推荐打开。
- **dbt-expectations / dbt-utils**：很多“常见自定义测试”已经有现成 package：`expression_is_true`、`recency`、`row_count_delta`。先看 package 再写自己的。
- **测试也是建模**：不要把所有约束都堆到 marts 层。Source / staging 层尽早测，越早发现越便宜。
- **CI 模式**：把测试拆 tag（`@nightly` / `@critical`），critical 在每个 PR 上跑，nightly 在定时任务跑。

## 思考题

- 写一个 generic test `recency`，参数是 `column_name` 和 `interval`，断言列里有过去 N 天的数据（避免数据停滞没人发现）。
- 在 `_marts.yml` 给 `fct_daily_rates.rate_change_pct` 加个 `expression_is_true: '>= -0.5 and <= 0.5'`，捕获异常的汇率跳变。需要装 dbt_expectations 包。
- `--store-failures` 在你公司怎么落地？谁负责定期看 audit 表？